# Compare RAG retrieval methods

Run from a checkout of this repository in Kaggle/Colab. This notebook demonstrates sparse BM25 and TF-IDF retrieval, dense Sentence Transformers + FAISS cosine search, hybrid reciprocal-rank fusion, optional cross-encoder reranking, and index persistence. Dense models download from Hugging Face on first use.

In [ ]:
%pip -q install -U sentence-transformers faiss-cpu numpy

In [ ]:
# Clone the repository if it is not already present in the runtime.
import os, sys
REPO_ROOT = "/kaggle/working/large-language-models" if os.path.exists("/kaggle/working/large-language-models/rag") else "/content/large-language-models"
if not os.path.isdir(os.path.join(REPO_ROOT, "rag")):
    import subprocess
    subprocess.run(["git", "clone", "https://github.com/MKarthik730/large-language-models.git", REPO_ROOT], check=True)
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)
from rag import RAGModel
documents = [
    "BM25 is a sparse lexical retrieval function based on term frequency and document length.",
    "TF-IDF weights terms by their frequency in a document and rarity across the corpus.",
    "Dense retrieval encodes text into vectors and finds semantically similar passages.",
    "FAISS provides efficient nearest-neighbor search for dense vector representations.",
    "Reciprocal rank fusion combines ranked lists without requiring comparable score scales.",
    "A cross-encoder reranker scores query and passage pairs for more precise ordering.",
    "Hybrid retrieval combines lexical matching with semantic embedding search.",
]
metadata = [{"source": f"demo-{index}.txt"} for index in range(len(documents))]
rag = RAGModel()
rag.add_documents(documents, metadata)
print("Indexed documents:", len(rag.documents))

In [ ]:
question = "How can I combine keyword and semantic search?"
for method in ("bm25", "tfidf", "dense", "hybrid"):
    print(f"\n--- {method.upper()} ---")
    for document in rag.retrieve(question, top_k=3, method=method):
        print(f"[{document.id}] {document.text}")
result = rag.query(question, method="hybrid", top_k=3)
print("\nPrompt for a separate generation model:\n", result["prompt"])
print("Sources:", result["sources"], "Scores:", result["scores"])

## Optional reranking

Cross-encoders require an additional model download and compute. This step is opt-in and may not fit a free-tier runtime; it reranks retrieved candidates rather than searching the entire corpus.

In [ ]:
# Uncomment to enable a cross-encoder reranker:
# reranked_rag = RAGModel(reranker_model="cross-encoder/ms-marco-MiniLM-L-6-v2")
# reranked_rag.add_documents(documents, metadata)
# for document in reranked_rag.retrieve(question, method="hybrid", rerank=True, top_k=3):
#     print(document.text)

In [ ]:
from pathlib import Path
index_prefix = str(Path("rag-demo-index"))
rag.save_index(index_prefix)
restored = RAGModel()
restored.load_index(index_prefix)
print("Restored docs:", len(restored.documents))
print("Search after reload:", [d.text for d in restored.retrieve(question, method="hybrid", top_k=2)])